# Extract MIF-ST embeddings — structure-source comparison

Re-runs MIF-ST embedding extraction for the **same 152 variants** using PDB
structures from **different sources** (AlphaFold2 vs trRosetta), to study how much
the structure source changes the resulting embeddings.

MIF-ST reads only the backbone N, Cα, C coordinates from each PDB, so it works
identically whether the structure came from AlphaFold2, trRosetta, or experiment —
but the embeddings it produces **differ by source**, which is exactly what we want
to measure here.

### How to use
1. Put this notebook in the same folder as `pretrained.py`, `pdb_utils.py`,
   `collaters.py`, `constants.py`, `utils.py` (the MIF-ST source files).
2. Set the paths in the **Config** cell for ONE source (e.g. AlphaFold2), run all.
3. Change the paths to the OTHER source (trRosetta), run again.
4. Use the **Verify** section to confirm the pipeline reproduces your existing
   embeddings before trusting the new ones.

> **Requires** `torch` and the MIF-ST dependencies. This is a separate task from the
> baseline notebook; you can reuse the same conda env.

## 1. Imports

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

from pretrained import load_model_and_alphabet
from pdb_utils import parse_PDB, process_coords

print("torch", torch.__version__)

torch 2.4.1+cpu


## 2. Config — edit here, once per structure source

Run the whole notebook once for AlphaFold2, then change these three paths to the
trRosetta versions and run again. Keep the output directories **separate** so the
two sets of embeddings do not overwrite each other.

In [2]:
MODEL    = "mifst"

# --- edit these three per source ---
CSV_PATH  = "data/demo_50_variants.csv"          # name / sequence / fitness / pdb / embed
PDB_DIR   = "data/PDB_AlphaFold/"                     # folder of .pdb files (trailing slash)
OUT_DIR   = "data/output_extract_mifst_AlphaFold/"            # where .pt embeddings are written
# for trRosetta run, e.g.:  PDB_DIR="PDB_trrosetta/"   OUT_DIR="output_extract_mif_tr/"

DEVICE = "cpu"   # MIF-ST is small; CPU is fine

os.makedirs(OUT_DIR, exist_ok=True)
print("PDB source :", PDB_DIR)
print("output to  :", OUT_DIR)

PDB source : data/PDB_AlphaFold/
output to  : data/output_extract_mifst_AlphaFold/


## 3. Sanity check — verify paths before the long run
Confirms the CSV loads, the source files exist, and a sample PDB parses.

In [3]:
problems = []
df = pd.read_csv(CSV_PATH)
for col in ["name", "sequence", "pdb", "embed"]:
    if col not in df.columns:
        problems.append(f"missing column '{col}' (got {list(df.columns)})")

# check a few PDB files exist
missing_pdb = [r["pdb"] for _, r in df.head(5).iterrows()
               if not os.path.exists(PDB_DIR + r["pdb"])]
if missing_pdb:
    problems.append(f"PDB files not found in {PDB_DIR}: {missing_pdb}")

# try parsing one
if not problems:
    coords, wt, _ = parse_PDB(PDB_DIR + df.iloc[0]["pdb"])
    print("parsed", df.iloc[0]["pdb"], "-> coords shape", coords.shape,
          "| seq length", len(wt))

if problems:
    print("PROBLEMS:")
    for p in problems: print("  -", p)
else:
    print(f"OK — {len(df)} variants, source ready.")

parsed E164Y.pdb -> coords shape (286, 3, 3) | seq length 286
OK — 50 variants, source ready.


## 4. Load MIF-ST model

In [4]:
print("Loading MIF-ST ...")
model, collater = load_model_and_alphabet(MODEL)
device = torch.device(DEVICE)
model = model.to(device).eval()
print("loaded on", device)

Loading MIF-ST ...
loaded on cpu


## 5. Extract embeddings
For each variant: parse PDB → build Cβ-based distance/angle features → run MIF-ST →
save the per-token embedding (L × 256) as `<name>_mifst_per_tok.pt` in `OUT_DIR`.
The `embed` column filenames are reused so downstream code finds them unchanged.

In [5]:
df = pd.read_csv(CSV_PATH).reset_index(drop=True)

skipped = []
with torch.no_grad(), tqdm(total=len(df)) as pbar:
    for _, row in df.iterrows():
        name, seq, pdb = row["name"], row["sequence"], row["pdb"]
        out_path = os.path.join(OUT_DIR, row["embed"])   # reuse embed filename

        try:
            coords, wt, _ = parse_PDB(PDB_DIR + pdb)
            coords = {"N": coords[:, 0], "CA": coords[:, 1], "C": coords[:, 2]}
            dist, omega, theta, phi = process_coords(coords)
            batch = [[seq,
                      torch.tensor(dist,  dtype=torch.float),
                      torch.tensor(omega, dtype=torch.float),
                      torch.tensor(theta, dtype=torch.float),
                      torch.tensor(phi,   dtype=torch.float)]]
            src, nodes, edges, connections, edge_mask = collater(batch)
            rep = model(src.to(device), nodes.to(device), edges.to(device),
                        connections.to(device), edge_mask.to(device),
                        result="repr")[0]
            torch.save(rep.detach().cpu(), out_path)   # per-token L x 256
        except Exception as e:
            skipped.append((name, str(e)))
        pbar.update(1)

print(f"done. wrote {len(df) - len(skipped)} embeddings to {OUT_DIR}")
if skipped:
    print("SKIPPED:")
    for n, e in skipped:
        print(f"  {n}: {e}")

100%|██████████| 50/50 [03:33<00:00,  4.28s/it]

done. wrote 50 embeddings to data/output_extract_mifst_AlphaFold/
